In [0]:
-- Definindo a tabela para o recorte temporal de pedidos
WITH tb_pedidos AS (
  SELECT *,
    to_date('2018-06-30', 'yyyy-MM-dd') AS DataReferencia
  FROM workspace.olist.orders
  WHERE order_purchase_timestamp < to_date('2018-07-01', 'yyyy-MM-dd')
),

-- Definindo a lista com os sellers para serem a base de construção da Feature Store
tb_Vendedores AS (
 SELECT DISTINCT
    tb2.seller_id,
    tb1.datareferencia
  FROM tb_pedidos as tb1
  INNER JOIN workspace.olist.order_items as tb2
    on tb1.order_id = tb2.order_id
),

-- Definindo a tabela com os produtos e suas categorias ao longo do tempo
tb_Produtos AS (
 SELECT 
    tb2.seller_id,
    tb3.product_category_name,
    tb3.product_id,
    tb1.order_purchase_timestamp,
    tb1.datareferencia,
    tb2.price,
    tb2.freight_value,
    tb3.product_weight_g/1000 as product_weight_kg,
    (tb3.product_height_cm * tb3.product_length_cm * tb3.product_width_cm) as product_volume_cm3
  FROM tb_pedidos as tb1
  INNER JOIN workspace.olist.order_items as tb2
    on tb1.order_id = tb2.order_id
  INNER JOIN workspace.olist.products as tb3
    on tb2.product_id = tb3.product_id
),

-- Criando as variáveis: Quantidade de categorias distintas no período* - vlCategoriasDistintas
tb01_DiverCatGrpProdutos AS (
    SELECT 
    datareferencia,
    seller_id,
    count(distinct case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_category_name end) as vlCategoriasDistintasD14,
    count(distinct case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_category_name end) as vlCategoriasDistintasD28,
    count(distinct case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_category_name end) as vlCategoriasDistintasD56,
    count(distinct case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_category_name end) as vlCategoriasDistintasD365,
    count(distinct product_category_name) as vlCategoriasDistintasVida
FROM tb_produtos
GROUP BY seller_id, datareferencia
ORDER BY vlCategoriasDistintasD14 DESC, vlCategoriasDistintasD28 DESC, vlCategoriasDistintasD56 DESC, vlCategoriasDistintasD365 DESC, vlCategoriasDistintasVida DESC
),

-- Criando as variáveis: Quantidade de produtos distintos no período* - vlProdutosDistintos
tb01_DiverCatProdutos AS (
    SELECT 
        datareferencia,
        seller_id,
        count(distinct case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_id end) as vlProdutosDistintosD14,
        count(distinct case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_id end) as vlProdutosDistintosD28,
        count(distinct case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_id end) as vlProdutosDistintosD56,
        count(distinct case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_id end) as vlProdutosDistintosD365,
        count(distinct product_id) as vlProdutosDistintosVida
    FROM tb_produtos
    GROUP BY seller_id, datareferencia
    ORDER BY vlProdutosDistintosD14 DESC, vlProdutosDistintosD28 DESC, vlProdutosDistintosD56 DESC, vlProdutosDistintosD365 DESC, vlProdutosDistintosvida DESC
),

-- Definindo a CTE para as variáveis de Peso dos Produtos Vendidos - Grupo 03
tb03_PesoProdutosVendidos AS (
    SELECT 
        datareferencia,
        seller_id,
        mean(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end) as vlMediaPesoProdutoD14,
        mean(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end) as vlMediaPesoProdutoD28,
        mean(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end) as vlMediaPesoProdutoD56,
        mean(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end) as vlMediaPesoProdutoD365,
        mean(product_weight_kg) as vlMediaPesoProdutoVida,
        median(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end) as vlMedianaPesoProdutoD14,
        median(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end) as vlMedianaPesoProdutoD28,
        median(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end) as vlMedianaPesoProdutoD56,
        median(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end) as vlMedianaPesoProdutoD365,
        median(product_weight_kg) as vlMedianaPesoProdutoVida,
        percentile(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end, 0.25) as vl25PesoProdutoD14,
        percentile(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end, 0.25) as vl25PesoProdutoD28,
        percentile(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end, 0.25) as vl25PesoProdutoD56,
        percentile(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end, 0.25) as vl25PesoProdutoD365,
        percentile(product_weight_kg, 0.25) as vl25PesoProdutoVida,
        percentile(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end, 0.75) as vl75PesoProdutoD14,
        percentile(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end, 0.75) as vl75PesoProdutoD28,
        percentile(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end, 0.75) as vl75PesoProdutoD56,
        percentile(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end, 0.75) as vl75PesoProdutoD365,
        percentile(product_weight_kg, 0.75) as vl75PesoProdutoVida,
        min(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end) as vlMinPesoProdutoD14,
        min(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end) as vlMinPesoProdutoD28,
        min(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end) as vlMinPesoProdutoD56,
        min(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end) as vlMinPesoProdutoD365,
        min(product_weight_kg) as vlMinPesoProdutoVida,
        max(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end) as vlMaxPesoProdutoD14,
        max(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end) as vlMaxPesoProdutoD28,
        max(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end) as vlMaxPesoProdutoD56,
        max(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end) as vlMaxPesoProdutoD365,
        max(product_weight_kg) as vlMaxPesoProdutoVida,
        sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end) as vlTotalPesoProdutosD14,
        sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end) as vlTotalPesoProdutosD28,
        sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end) as vlTotalPesoProdutosD56,
        sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end) as vlTotalPesoProdutosD365,
        sum(product_weight_kg) as vlTotalPesoProdutosVida
    FROM tb_produtos
    GROUP BY datareferencia, seller_id
),

-- Definindo a CTE para as variáveis de Cubagem dos Produtos Vendidos - Grupo 04
tb04_CubagemProdutosVendidos AS (
    SELECT 
        datareferencia,
        seller_id,
        mean(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_volume_cm3 end) as vlMediaCubagemProdutosD14,
        mean(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_volume_cm3 end) as vlMediaCubagemProdutosD28,
        mean(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_volume_cm3 end) as vlMediaCubagemProdutosD56,
        mean(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_volume_cm3 end) as vlMediaCubagemProdutosD365,
        mean(product_volume_cm3) as vlMediaCubagemProdutosVida,
        sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_volume_cm3 end) as vlTotalCubagemProdutosD14,
        sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_volume_cm3 end) as vlTotalCubagemProdutosD28,
        sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_volume_cm3 end) as vlTotalCubagemProdutosD56,
        sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_volume_cm3 end) as vlTotalCubagemProdutosD365,
        sum(product_volume_cm3) as vlTotalCubagemProdutosVida
    FROM tb_produtos
    GROUP BY datareferencia, seller_id
),

-- Definindo a CTE para as variáveis de Preco e Frete por kg dos Produtos Vendidos - Grupo 05
tb05_IndicadoresKgProdVendidos AS (
    SELECT 
        datareferencia,
        seller_id,
        (sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then price end) / sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end)) as vlPrecoKgD14,
        (sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then price end) / sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end)) as vlPrecoKgD28,
        (sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then price end) / sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end)) as vlPrecoKgD56,
        (sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then price end) / sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end)) as vlPrecoKgD365,
        (sum(price) / sum(product_weight_kg)) as vlPrecoKgVida,
        (sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then freight_value end) / sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 14 then product_weight_kg end)) as vlFreteKgD14,
        (sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then freight_value end) / sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 28 then product_weight_kg end)) as vlFreteKgD28,
        (sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then freight_value end) / sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 56 then product_weight_kg end)) as vlFreteKgD56,
        (sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then freight_value end) / sum(case when datediff(DataReferencia, order_purchase_timestamp) <= 365 then product_weight_kg end)) as vlFreteKgD365,
        (sum(freight_value) / sum(product_weight_kg)) as vlFreteKgVida
    FROM tb_produtos
    GROUP BY datareferencia, seller_id
),

-- Definindo a CTE para as variáveis das estatisticas de Caracterização e fotos do portfólio de produtos - Grupo 07 etapa 1
tb07_A_AtrProdutosPortfolio AS (
 SELECT DISTINCT
    tb2.seller_id,
    tb3.product_id,
    tb1.datareferencia,
    coalesce(tb3.product_description_lenght, 0) as product_description_lenght,
    coalesce(tb3.product_photos_qty, 0) as product_photos_qty,
    coalesce(tb3.product_weight_g, 0)/1000 as product_weight_kg,
    (coalesce(tb3.product_height_cm, 0) * coalesce(tb3.product_length_cm, 0) * coalesce(tb3.product_width_cm, 0)) as product_volume_cm3
  FROM tb_pedidos as tb1
  INNER JOIN workspace.olist.order_items as tb2
    on tb1.order_id = tb2.order_id
  INNER JOIN workspace.olist.products as tb3
    on tb2.product_id = tb3.product_id
ORDER BY tb2.seller_id, product_description_lenght
),

-- Criando as CTE com as estatisticas da descrição e das fotos do portfólio de produtos - Grupo 07 etapa 2
tb07_B_AtrProdutosPortfolio as (
SELECT 
  datareferencia, 
  seller_id,
  mean(product_description_lenght) as vlMediaCaracteresDescricao,
  median(product_description_lenght) as vlMedianaCaracteresDescricao,
  percentile(product_description_lenght, 0.25) as vl25CaracteresDescricao,
  percentile(product_description_lenght, 0.75) as vl75CaracteresDescricao,
  min(product_description_lenght) as vlMinCaracteresDescricao,
  max(product_description_lenght) as vlMaxCaracteresDescricao,
  mean(product_photos_qty) as vlMediaFotosProdutos
FROM tb07_A_AtrProdutosPortfolio
GROUP BY seller_id, datareferencia
ORDER BY seller_id, datareferencia
),

-- Criando as CTE com as estatisticas do peso do portfólio de produtos - Grupo 08
tb08_PesoProdutosPortfolio as (
SELECT 
  datareferencia, 
  seller_id,
  mean(product_weight_kg) as vlMediaPesoPortfolio,
  median(product_weight_kg) as vlMedianaPesoPortfolio,
  percentile(product_weight_kg, 0.25) as vl25PesoPortfolio,
  percentile(product_weight_kg, 0.75) as vl75PesoPortfolio,
  min(product_weight_kg) as vlMinPesoPortfolio,
  max(product_weight_kg) as vlMaxPesoPortfolio,
  sum(product_weight_kg) as vlTotalPesoPortfolio
FROM tb07_A_AtrProdutosPortfolio
GROUP BY seller_id, datareferencia
ORDER BY seller_id, datareferencia
),

--  Criando a Tabela da Feature Store de Produtos
tb_FS_Produtos AS (
    SELECT 
        tb0.datareferencia,
        tb0.seller_id,
        tb1.vlCategoriasDistintasD14,
        tb1.vlCategoriasDistintasD28,
        tb1.vlCategoriasDistintasD56,
        tb1.vlCategoriasDistintasD365,
        tb1.vlCategoriasDistintasvida,
        tb2.vlProdutosDistintosD14,
        tb2.vlProdutosDistintosD28,
        tb2.vlProdutosDistintosD56,
        tb2.vlProdutosDistintosD365,
        tb2.vlProdutosDistintosVida,
        tb3.vlMediaPesoProdutoD14,
        tb3.vlMediaPesoProdutoD28,
        tb3.vlMediaPesoProdutoD56,
        tb3.vlMediaPesoProdutoD365,
        tb3.vlMediaPesoProdutoVida,
        tb3.vlMedianaPesoProdutoD14,
        tb3.vlMedianaPesoProdutoD28,
        tb3.vlMedianaPesoProdutoD56,
        tb3.vlMedianaPesoProdutoD365,
        tb3.vlMedianaPesoProdutoVida,
        tb3.vl25PesoProdutoD14,
        tb3.vl25PesoProdutoD28,
        tb3.vl25PesoProdutoD56,
        tb3.vl25PesoProdutoD365,
        tb3.vl25PesoProdutoVida,
        tb3.vl75PesoProdutoD14,
        tb3.vl75PesoProdutoD28,
        tb3.vl75PesoProdutoD56,
        tb3.vl75PesoProdutoD365,
        tb3.vl75PesoProdutoVida,
        tb3.vlMinPesoProdutoD14,
        tb3.vlMinPesoProdutoD28,
        tb3.vlMinPesoProdutoD56,
        tb3.vlMinPesoProdutoD365,
        tb3.vlMinPesoProdutoVida,
        tb3.vlMaxPesoProdutoD14,
        tb3.vlMaxPesoProdutoD28,
        tb3.vlMaxPesoProdutoD56,
        tb3.vlMaxPesoProdutoD365,
        tb3.vlMaxPesoProdutoVida,
        tb3.vlTotalPesoProdutosD14,
        tb3.vlTotalPesoProdutosD28,
        tb3.vlTotalPesoProdutosD56,
        tb3.vlTotalPesoProdutosD365,
        tb3.vlTotalPesoProdutosVida,
        tb4.vlMediaCubagemProdutosD14,
        tb4.vlMediaCubagemProdutosD28,
        tb4.vlMediaCubagemProdutosD56,
        tb4.vlMediaCubagemProdutosD365,
        tb4.vlMediaCubagemProdutosVida,
        tb4.vlTotalCubagemProdutosD14,
        tb4.vlTotalCubagemProdutosD28,
        tb4.vlTotalCubagemProdutosD56,
        tb4.vlTotalCubagemProdutosD365,
        tb4.vlTotalCubagemProdutosVida,
        tb5.vlPrecoKgD14,
        tb5.vlPrecoKgD28,
        tb5.vlPrecoKgD56,
        tb5.vlPrecoKgD365,
        tb5.vlPrecoKgVida,
        tb5.vlFreteKgD14,
        tb5.vlFreteKgD28,
        tb5.vlFreteKgD56,
        tb5.vlFreteKgD365,
        tb5.vlFreteKgVida,
        tb7.vlMediaCaracteresDescricao,
        tb7.vlMedianaCaracteresDescricao,
        tb7.vl25CaracteresDescricao,
        tb7.vl75CaracteresDescricao,
        tb7.vlMinCaracteresDescricao,
        tb7.vlMaxCaracteresDescricao,
        tb7.vlMediaFotosProdutos,
        tb8.vlMediaPesoPortfolio,
        tb8.vlMedianaPesoPortfolio,
        tb8.vl25PesoPortfolio,
        tb8.vl75PesoPortfolio,
        tb8.vlMinPesoPortfolio,
        tb8.vlMaxPesoPortfolio,
        tb8.vlTotalPesoPortfolio
    FROM tb_vendedores as tb0
    LEFT JOIN tb01_DiverCatGrpProdutos AS tb1
        ON tb0.datareferencia = tb1.datareferencia
        AND tb0.seller_id = tb1.seller_id
    LEFT JOIN tb01_DiverCatprodutos as tb2
        ON tb0.datareferencia = tb2.datareferencia 
        AND tb0.seller_id = tb2.seller_id
    LEFT JOIN tb03_pesoprodutosvendidos AS tb3
        ON tb0.datareferencia = tb3.datareferencia
        AND tb0.seller_id = tb3.seller_id
    LEFT JOIN tb04_cubagemprodutosvendidos AS tb4
        ON tb0.datareferencia = tb4.datareferencia
        AND tb0.seller_id = tb4.seller_id
    LEFT JOIN tb05_indicadoreskgprodvendidos AS tb5
        ON tb0.datareferencia = tb5.datareferencia
        AND tb0.seller_id = tb5.seller_id
    LEFT JOIN tb07_b_atrprodutosportfolio AS tb7
        ON tb0.datareferencia = tb7.datareferencia
        AND tb0.seller_id = tb7.seller_id
    LEFT JOIN tb08_pesoprodutosportfolio AS tb8
        ON tb0.datareferencia = tb8.datareferencia
        AND tb0.seller_id = tb8.seller_id
)

SELECT *
FROM tb_fs_produtos

